# Phases 3R–7 in one Kaggle run

This notebook overlaps the corrected CPU probe run with the GPU VQA run, then automatically executes the dependent joint and causal phases. It never evaluates the official CUB test split. Run all three cells in order with a GPU and internet enabled. Completed per-decision records resume after interruption.


In [ ]:
# Setup, verify the latest branch, and locate the three existing inputs.
import os, sys, json, subprocess, hashlib, zipfile
from pathlib import Path
from IPython.display import display, Markdown, FileLink

PROJECT = Path('/kaggle/working/newpipeline/projects/logit_evidence_routing')
os.chdir(PROJECT)
def git(*args): return subprocess.check_output(['git', *args], text=True).strip()
if git('status', '--porcelain', '--untracked-files=no'):
    raise RuntimeError('Preserve tracked repository edits before updating.')
subprocess.run(['git','fetch','--no-tags','origin','+refs/heads/feat/iclr:refs/remotes/origin/feat/iclr'], check=True)
subprocess.run(['git','switch','feat/iclr'], check=True)
subprocess.run(['git','merge','--ff-only','origin/feat/iclr'], check=True)
HEAD = git('rev-parse','HEAD')
print('Commit:', HEAD)
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','-r','requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'], check=True, env=dict(os.environ, PYTHONPATH='src'))

STAGE_CACHE = Path('/kaggle/working/phase2_stage_cache')
if not STAGE_CACHE.is_dir():
    candidates = list(Path('/kaggle/input').rglob('phase2_stage_cache/validation_report.json'))
    if len(candidates) != 1: raise FileNotFoundError('Set STAGE_CACHE to the completed Phase 2 cache.')
    STAGE_CACHE = candidates[0].parent
review = json.loads((PROJECT/'reports/development_20260911/phase4_review_decision.json').read_text())
def file_sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
required_phase4_files = ('bundle_manifest.json','phase4_validation_report.json','phase4_run_report.json','attribute_eligibility.csv','attribute_metrics.csv','attribute_macro_summary.csv')
def valid_phase4_dir(path):
    return (path.is_dir() and all((path/name).is_file() for name in required_phase4_files)
            and file_sha(path/'bundle_manifest.json') == review['bundle_manifest_sha256'])
phase4_candidates = []
for root in (Path('/kaggle/input'), Path('/kaggle/working')):
    for manifest in root.rglob('bundle_manifest.json'):
        if valid_phase4_dir(manifest.parent): phase4_candidates.append(manifest.parent)
# If the downloaded result bundle was uploaded as a ZIP, verify its embedded manifest before extraction.
phase4_zips = [p for root in (Path('/kaggle/input'), Path('/kaggle/working')) for p in root.rglob('*.zip') if 'phase4' in p.name.casefold()]
extracted_phase4 = Path('/kaggle/working')/f"reviewed_phase4_{review['bundle_manifest_sha256'][:12]}"
for archive in sorted(phase4_zips):
    try:
        with zipfile.ZipFile(archive) as opened:
            manifests = [name for name in opened.namelist() if Path(name).name == 'bundle_manifest.json']
            if not any(hashlib.sha256(opened.read(name)).hexdigest() == review['bundle_manifest_sha256'] for name in manifests):
                continue
            extracted_phase4.mkdir(parents=True, exist_ok=True)
            for item in opened.infolist():
                target = (extracted_phase4/item.filename).resolve()
                if target != extracted_phase4.resolve() and extracted_phase4.resolve() not in target.parents:
                    raise RuntimeError(f'Unsafe ZIP member: {item.filename}')
            opened.extractall(extracted_phase4)
            break
    except zipfile.BadZipFile:
        continue
for manifest in extracted_phase4.rglob('bundle_manifest.json') if extracted_phase4.is_dir() else ():
    if valid_phase4_dir(manifest.parent): phase4_candidates.append(manifest.parent)
phase4_candidates = sorted({path.resolve() for path in phase4_candidates}, key=lambda path: (0 if str(path).startswith('/kaggle/working/') else 1, str(path)))
if not phase4_candidates:
    raise FileNotFoundError('Reviewed Phase 4 bundle not found. Searched /kaggle/input and /kaggle/working, including Phase 4 ZIPs: '+repr([str(p) for p in phase4_zips]))
PHASE4_DIR = phase4_candidates[0]
stage_cfg = json.loads((STAGE_CACHE/'run_config.json').read_text())
CUB_ROOT = Path(stage_cfg['dataset']['root'])
if not CUB_ROOT.is_dir():
    sys.path.insert(0, str(PROJECT/'src'))
    from lger.cub import discover_cub_root
    CUB_ROOT = discover_cub_root(Path('/kaggle/input'))
PARTS = CUB_ROOT/'parts/parts.txt'
OUT = Path('/kaggle/working')/f'multiphase_development_{HEAD[:12]}'
print('Stage cache:', STAGE_CACHE)
print('Phase 4:', PHASE4_DIR)
print('CUB:', CUB_ROOT)
print('Output:', OUT)


In [ ]:
# One command: Phase 3R || Phase 5, followed by Phase 6 -> Phase 7 -> freeze -> paper package.
os.chdir(PROJECT)
command = [sys.executable, 'scripts/run_multiphase_development.py',
           '--stage-cache', str(STAGE_CACHE),
           '--phase4-dir', str(PHASE4_DIR),
           '--cub-root', str(CUB_ROOT),
           '--part-vocabulary', str(PARTS),
           '--output-root', str(OUT),
           '--phase3-device', 'cpu',
           '--phase7-max-per-outcome-per-attribute', '5']
print('$', ' '.join(command), flush=True)
subprocess.run(command, check=True, env=dict(os.environ, PYTHONPATH='src', PYTHONUNBUFFERED='1', CUBLAS_WORKSPACE_CONFIG=':4096:8'))


In [ ]:
# Show the scientific gate and package the paper-ready tables.
status = json.loads((OUT/'multiphase_status.json').read_text())
display(Markdown('## Multiphase status\n\n```json\n'+json.dumps(status, indent=2)+'\n```'))
display(Markdown((OUT/'phase6/INTERMEDIATE_FINDINGS.md').read_text()))
display(Markdown((OUT/'paper_package/PAPER_FINDINGS.md').read_text()))
archive = subprocess.check_output([sys.executable,'-c',
    "import shutil; print(shutil.make_archive(r'"+str(OUT/'paper_results_package')+"','zip',r'"+str(OUT/'paper_package')+"'))"], text=True).strip()
display(FileLink(archive))
display(FileLink(str(OUT/'multiphase_status.json')))
print('Official test images used:', status['official_test_images_used'])
